In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# แนะนำ primitives

<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</AccordionItem>
</Accordion>
<span id="qpu-access-patterns"></span>
## ทำไม Qiskit ถึงแนะนำ primitives?
เช่นเดียวกับยุคแรกของคอมพิวเตอร์คลาสสิก ที่นักพัฒนาต้องจัดการ CPU register โดยตรง อินเทอร์เฟซแรกสุดของ QPU ก็แค่ส่งคืนข้อมูลดิบจากอุปกรณ์ควบคุมอิเล็กทรอนิกส์
เรื่องนี้ไม่ใช่ปัญหาใหญ่เมื่อ QPU ยังอยู่แค่ในห้องปฏิบัติการและให้เฉพาะนักวิจัยเข้าถึงโดยตรง
เพื่อตอบโจทย์ที่นักพัฒนาส่วนใหญ่ไม่จำเป็นต้องรู้วิธีแปลงข้อมูลดิบเหล่านั้นเป็น 0 และ 1 Qiskit จึงแนะนำ `backend.run` ซึ่งเป็น abstraction แรกสำหรับการเข้าถึง QPU บนคลาวด์ ทำให้นักพัฒนาทำงานกับรูปแบบข้อมูลที่คุ้นเคยและโฟกัสกับภาพรวมได้มากขึ้น

เมื่อการเข้าถึง QPU แพร่หลายมากขึ้น และมีการพัฒนา quantum algorithm ใหม่ๆ มากมาย ความต้องการ abstraction ระดับสูงขึ้นก็เกิดขึ้นอีกครั้ง Qiskit จึงแนะนำอินเทอร์เฟซ primitives ที่ถูกออปติไมซ์สำหรับสองงานหลักในการพัฒนา quantum algorithm: การประมาณค่า expectation value (`Estimator`) และการสุ่มตัวอย่างจาก Circuit (`Sampler`) เป้าหมายคือช่วยให้นักพัฒนาโฟกัสที่นวัตกรรมมากขึ้นและลดเวลาที่ต้องใช้กับการแปลงข้อมูล อินเทอร์เฟซ primitives แทนที่อินเทอร์เฟซ `backend.run` เนื่องจาก `Sampler` ให้การเข้าถึง hardware โดยตรงแบบเดียวกับที่ `backend.run` เคยให้ไว้

## primitive คืออะไร?
ระบบคอมพิวเตอร์ถูกสร้างขึ้นบน abstraction หลายชั้น abstraction ช่วยให้โฟกัสที่ระดับรายละเอียดที่เกี่ยวข้องกับงานนั้นๆ ได้ ยิ่งใกล้ hardware มากขึ้นเท่าไหร่ ระดับ abstraction ที่ต้องการก็ยิ่งต่ำลง (เช่น อาจต้องย้ายหรือจัดการข้อมูลในระดับคำสั่ง CPU) ยิ่งงานที่ต้องทำซับซ้อนมากขึ้นเท่าไหร่ abstraction ก็จะสูงขึ้น (เช่น อาจใช้ programming library สำหรับการคำนวณพีชคณิต)

ในบริบทนี้ *primitive* คือคำสั่งประมวลผลที่เล็กที่สุด บล็อคสร้างที่เรียบง่ายที่สุดที่สามารถสร้างสิ่งที่มีประโยชน์สำหรับระดับ abstraction ที่กำหนด

ความก้าวหน้าล่าสุดใน quantum computing ได้เพิ่มความจำเป็นในการทำงานที่ระดับ abstraction ที่สูงขึ้น เมื่อสาขานี้มุ่งไปสู่ quantum processing unit (QPU) ขนาดใหญ่และ workflow ที่ซับซ้อนขึ้น จุดโฟกัสจึงเปลี่ยนจากการโต้ตอบกับสัญญาณ Qubit แต่ละตัวไปสู่การมอง quantum device เป็นระบบที่ทำงานตามที่ต้องการ

สองงานที่พบบ่อยที่สุดสำหรับ quantum computer คือการสุ่มตัวอย่าง quantum state และการคำนวณ expectation value งานทั้งสองนี้เป็นแรงบันดาลใจในการออกแบบ Qiskit primitives: **Estimator** และ **Sampler**

- Estimator คำนวณ expectation value ของ observable เทียบกับ state ที่เตรียมโดย quantum circuit
- Sampler สุ่มตัวอย่างจาก output register จากการรัน quantum circuit

โดยสรุป โมเดลการคำนวณที่ Qiskit primitives แนะนำนั้นนำ quantum programming เข้าใกล้ classical programming ในปัจจุบันมากขึ้น ซึ่งโฟกัสน้อยลงที่รายละเอียด hardware และมากขึ้นที่ผลลัพธ์ที่ต้องการ

## นิยามและการ implement primitives
Qiskit primitives มีสองประเภท: base class และ implementation ต่างๆ Estimator และ Sampler primitives ถูกนิยามโดย primitive base class แบบ open-source ที่อยู่ใน Qiskit SDK (ใน module [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives)) Provider ต่างๆ (เช่น Qiskit Runtime) สามารถใช้ base class เหล่านี้เพื่อพัฒนา Sampler และ Estimator ของตนเอง ผู้ใช้ส่วนใหญ่จะโต้ตอบกับ provider implementation ไม่ใช่ base primitives โดยตรง

### Base class
`Base` primitives เป็น abstract class ที่นิยาม interface ทั่วไปสำหรับการ implement primitives คลาสอื่นๆ ทั้งหมดใน module [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives) สืบทอดจาก base class เหล่านี้ นักพัฒนาควรใช้สิ่งเหล่านี้หากสนใจสร้างโมเดลการรันแบบ primitives-based ของตนเองสำหรับ provider เฉพาะ class เหล่านี้อาจมีประโยชน์สำหรับผู้ที่ต้องการการประมวลผลแบบ custom สูงและพบว่า implementation primitives ที่มีอยู่ง่ายเกินไปสำหรับความต้องการของตน ผู้ใช้ทั่วไปจะไม่ใช้ base class โดยตรง

[`BaseEstimatorV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV1) และ [`BaseSamplerV1`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV1) - แม้ว่า V1 primitives ยังสามารถใช้งานได้ คู่มือเหล่านี้มุ่งเน้นที่ V2 primitives เนื่องจากเป็นเวอร์ชันล่าสุดและใช้กันทั่วไปมากกว่า

[`BaseEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseEstimatorV2) และ [`BaseSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BaseSamplerV2) - Qiskit reference primitives ปฏิบัติตามข้อกำหนด interface เหล่านี้

<span id="implementations"></span>
### Implementations
primitives ทั้งหมดถูกสร้างจาก base class ดังนั้นจึงมีโครงสร้างและการใช้งานทั่วไปแบบเดียวกัน ตัวอย่างเช่น รูปแบบ input สำหรับ Estimator primitives ทั้งหมดเหมือนกัน อย่างไรก็ตาม มีความแตกต่างใน implementation ที่ทำให้แต่ละอันมีเอกลักษณ์

นี่คือ implementation ของ primitives base class:

- [Qiskit Runtime primitives](/guides/qiskit-runtime-primitives) ได้แก่ [`EstimatorV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) และ [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2) ให้ implementation ที่ซับซ้อนกว่า (เช่น รวม error mitigation) ในรูปแบบบริการบนคลาวด์ implementation นี้ของ base primitives ใช้สำหรับเข้าถึง IBM Quantum&reg; hardware

- [`StatevectorEstimator`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorEstimator) และ [`StatevectorSampler`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.StatevectorSampler#statevectorsampler) - Reference implementation ของ primitives ที่ใช้ simulator ในตัวของ Qiskit สร้างด้วย Qiskit module [`quantum_info`](https://docs.quantum.ibm.com/api/qiskit/quantum_info#quantum-information) ผลิตผลลัพธ์จากการจำลอง statevector ที่สมบูรณ์แบบ เข้าถึงผ่าน Qiskit ดู [Exact simulation with Qiskit primitives](/guides/simulate-with-qiskit-sdk-primitives) สำหรับรายละเอียดการใช้งาน

- [`BackendEstimatorV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendEstimatorV2) และ [`BackendSamplerV2`](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.BackendSamplerV2) - ใช้ class เหล่านี้เพื่อ "ห่อ" ทรัพยากร quantum computing ใดก็ได้ให้เป็น primitive ทำให้เขียนโค้ดแบบ primitive-style สำหรับ provider ที่ยังไม่มีอินเทอร์เฟซแบบ primitives-based ใช้งาน class เหล่านี้เหมือน Sampler และ Estimator ปกติ ยกเว้นต้องระบุอาร์กิวเมนต์ `backend` เพิ่มเติมเพื่อเลือก quantum computer ที่ต้องการรัน เข้าถึงโดยใช้ Qiskit ดูคู่มือ [backend primitives](/guides/get-started-with-backend-primitives) สำหรับข้อมูลเพิ่มเติม

## Options
คุณสามารถส่ง options ให้ primitives เพื่อปรับแต่งตามความต้องการ แม้ว่า interface ของเมธอด `run()` ของ primitives จะเหมือนกันในทุก implementation แต่ options ของแต่ละ implementation ไม่เหมือนกัน ดูข้อมูลอ้างอิง API สำหรับ primitive implementation เฉพาะเพื่อเรียนรู้เกี่ยวกับ options ที่รองรับ

ตัวอย่างเช่น ดูหัวข้อ [Estimator options](/guides/estimator-options) และ [Sampler options](/guides/sampler-options) เพื่อเรียนรู้เกี่ยวกับ options สำหรับ Qiskit Runtime primitives หรือดู [Qiskit Aer API references](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html) สำหรับ options ของ Qiskit Aer primitives

## ประโยชน์ของ Qiskit primitives
ด้วย primitives ผู้ใช้ Qiskit สามารถเขียน quantum code สำหรับ QPU เฉพาะโดยไม่ต้องจัดการทุกรายละเอียดอย่างชัดเจน นอกจากนี้ เนื่องจาก abstraction ชั้นเพิ่มเติม อาจเข้าถึงความสามารถ hardware ขั้นสูงของ provider ที่กำหนดได้ง่ายขึ้น ตัวอย่างเช่น ด้วย Qiskit Runtime primitives สามารถใช้ประโยชน์จากความก้าวหน้าล่าสุดใน error mitigation และ suppression โดยการสลับตัวเลือกเช่น [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level) ของ primitive แทนที่จะสร้าง implementation ของเทคนิคเหล่านี้เอง

สำหรับ hardware provider การ implement primitives โดยตรงหมายความว่าให้วิธีที่ "ใช้งานได้ทันที" มากขึ้นสำหรับผู้ใช้เพื่อเข้าถึงฟีเจอร์ hardware เช่น เทคนิคการประมวลผลขั้นสูง จึงทำให้ผู้ใช้ได้รับประโยชน์จากความสามารถที่ดีที่สุดของ hardware ได้ง่ายขึ้น

## ขั้นตอนถัดไป
> **Tip:** - ทำความเข้าใจ [primitive input and output](/guides/primitive-input-output)
> - ดูตัวอย่างรายละเอียด [examples](/guides/simulate-with-qiskit-sdk-primitives)
> - ฝึกกับ primitives ด้วยการเรียน [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) ใน IBM Quantum Learning
> - ดู [Create a provider](/guides/create-a-provider) เพื่อเรียนรู้วิธี implement Sampler และ Estimator primitives ของตนเอง
> - ดู [API references](https://docs.quantum.ibm.com/api/qiskit/primitives)
> - อ่าน [Migrate to V2 primitives](/guides/v2-primitives)
> - เรียนรู้เกี่ยวกับ [Qiskit Runtime primitives](/guides/qiskit-runtime-primitives) ที่ใช้สำหรับรัน circuit บน IBM QPU